In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def beam_splitter(r):
    """
    Returns the beam splitter matrix.

    Args:
        - r (float): The reflection coefficient of the beam splitter.
    Returns:
        - (np.array): 2 x 2 matrix that represents the beam
        splitter matrix.    
    """
    t = np.sqrt(1 - r**2)
    return np.array([[r, t], [t, -r]])


dev = qml.device('default.qubit', wires=1, shots=None)

@qml.qnode(dev)
def mz_interferometer(r):
    """
    This QNode returns the probability that either A or C
    detect a photon, and the probability that D detects a photon.
    
    Args:
        - r (float): The reflection coefficient of the beam splitters.
    Returns: 
        - np.array(float): An array of shape (2,), where the first 
        element is the probability of detection at A or C,
        and the second element is the probability of detection at D.
    """
    # Initial state is |0> (photon in path 0)
    # default.qubit device initializes all wires to |0>

    # First beam splitter
    qml.QubitUnitary(beam_splitter(r), wires=0)
    
    # Mid-circuit measurement for Detector A
    # m0 == 0 means photon is in path 0 (A detects)
    # m0 == 1 means photon is in path 1 (A does not detect)
    m0 = qml.measure(0)
    
    @qml.cond(m0 == 1)
    def second_stage():
        # Apply the second beam splitter only if A did not detect (photon is in path 1)
        qml.QubitUnitary(beam_splitter(r), wires=0)

    # Execute the conditional stage
    second_stage()

    # qml.probs(wires=[0]) computes the total probabilities of the qubit
    # ending in |0> or |1>, summing over all possible classical measurement outcomes
    # of m0.
    #
    # Let P(event) denote the probability of an event.
    # P(A detects) = r^2 (photon in path 0 after 1st BS). This contributes to p_AC.
    # P(A does NOT detect) = (1-r^2) (photon in path 1 after 1st BS).
    #
    # If A does NOT detect, then the second BS is applied to the |1> state.
    # The state becomes T|0> - R|1>.
    #
    # P(C detects) = P(A does NOT detect) * P(photon in path 0 after 2nd BS | A did NOT detect)
    #              = (1-r^2) * T^2 = (1-r^2) * (1-r^2)
    # This contributes to p_AC.
    #
    # P(D detects) = P(A does NOT detect) * P(photon in path 1 after 2nd BS | A did NOT detect)
    #              = (1-r^2) * (-R)^2 = (1-r^2) * r^2
    # This contributes to p_D.
    #
    # Therefore:
    # p_AC = P(A detects) + P(C detects) = r^2 + (1-r^2)^2
    # p_D = P(D detects) = (1-r^2) * r^2
    #
    # qml.probs(wires=[0])[0] will be P(final state is |0>), which sums:
    #   - The probability that m0=0 (A detects), where the effective "final state" is |0>. This is r^2.
    #   - The probability that m0=1 AND the second BS puts it in |0> (C detects). This is (1-r^2)*(1-r^2).
    # So, qml.probs(wires=[0])[0] = r^2 + (1-r^2)^2 = p_AC.
    #
    # qml.probs(wires=[0])[1] will be P(final state is |1>), which sums:
    #   - The probability that m0=0 (A detects), which does not contribute to |1>. This is 0.
    #   - The probability that m0=1 AND the second BS puts it in |1> (D detects). This is (1-r^2)*r^2.
    # So, qml.probs(wires=[0])[1] = (1-r^2)*r^2 = p_D.

    final_probabilities = qml.probs(wires=[0])

    return final_probabilities


def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = mz_interferometer(ins).tolist()
    
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, atol=1e-4), "Not the correct probabilities"
